In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ast
from astropy.table import Table
from scipy import stats
from fbm import FBM
from scipy.stats import normaltest, ks_2samp, ttest_ind, zscore

In [2]:
df = Table.read("DATA_TRACK_B.csv")

In [3]:
exp = df["exp"]
strain = df["strain"]
fov = df["fov"]
Id = df["Id"]
l = df["l"]
w = df["w"]
v = df["v"]
v0 = df["v0"]
vf = df["vf"]
grexp = df["grexp"]
grinst = df["grinst"]
tfrombirth = df["tfrombirth"]
ccphase = df["ccphase"]
tau = df["tau"]
t0 = df["t0"]
tf = df["tf"]
ttraj = df["ttraj"]
xtraj = df["xtraj"]
ytraj = df["ytraj"]
trajlabel = df["trajlabel"]

## Dati traiettorie -> array ||FUNZIONA||

In [4]:
for i in range(0, len(ytraj)):
    ttraj[i] = ttraj[i][1:-1]
    xtraj[i] = xtraj[i][1:-1]
    ytraj[i] = ytraj[i][1:-1]

In [5]:
t_bid = []

t_bid.append(np.array(ttraj[i].split(", "), dtype=float))
x_bid = [dato.split(", ") for dato in xtraj]
y_bid = [dato.split(", ") for dato in ytraj]

In [49]:
t_array = np.array(t_bid, dtype=np.float64)
x_array = np.array(x_bid, dtype=np.float64)
y_array = np.array(y_bid, dtype=np.float64)

x_first = x_array[:,0]
y_first = y_array[:,0]

x_array = x_array - x_first[:, np.newaxis]
y_array = y_array - y_first[:, np.newaxis]

Separo per strain (ori e ter)

In [50]:
ori_mask = strain == 'EO'
ter_mask = strain == 'ET'

x_ori = x_array[ori_mask]
y_ori = y_array[ori_mask]

x_ter = x_array[ter_mask]
y_ter = y_array[ter_mask]

## Funzione per simulare taaante traiettorie subdiffusive come voglio io

### Fractional Brownian Motion (fBM) ||FUNZIONA||

In [70]:
def simulate_fbm_trajectories(n_particles=2500, n_steps=40, hurst=0.15, length=1.0,
                               random_seed=42, initial_position_range=1.0):
    np.random.seed(random_seed)

    x = np.zeros((n_particles, n_steps))
    y = np.zeros((n_particles, n_steps))

    for i in range(n_particles):
        f_x = FBM(n=n_steps-1, hurst=hurst, length=length, method='daviesharte')
        f_y = FBM(n=n_steps-1, hurst=hurst, length=length, method='daviesharte')

        fbm_x = f_x.fbm()
        fbm_y = f_y.fbm()

        init_x = np.random.uniform(-initial_position_range, initial_position_range)
        init_y = np.random.uniform(-initial_position_range, initial_position_range)

        x[i] = fbm_x + init_x
        y[i] = fbm_y + init_y

        x[i, 0] = init_x
        y[i, 0] = init_y

    return x, y

x_fbm, y_fbm = simulate_fbm_trajectories()

### Continuous-Time Random Walk (CTRW) ||FUNZIONA||

In [71]:
def simulate_CTRW(n_particles=2500, n_steps=40, alpha=0.3, step_scale=1.0):
    x = np.zeros((n_particles, n_steps))
    y = np.zeros((n_particles, n_steps))

    for i in range(n_particles):
        positions_x = [0]
        positions_y = [0]

        for _ in range(1, n_steps):
            # Salto spaziale (normale, come nel Browniano)
            dx, dy = np.random.normal(0, step_scale, 2)

            positions_x.append(positions_x[-1] + np.sqrt(2*alpha)*dx)
            positions_y.append(positions_y[-1] + np.sqrt(2*alpha)*dy)

        x[i] = positions_x
        y[i] = positions_y

    return np.array(x), np.array(y)

x_ctrw, y_ctrw = simulate_CTRW()

# Moti diffusivi

## Pareti cellulari ||FUNZIONA||

In [72]:
length = np.mean(l)
width = np.mean(w)

cell_x = [-length/2, length/2, length/2, -length/2, -length/2]
cell_y = [-width/2, -width/2, width/2, width/2, -width/2]

length_ori = np.mean(l[ori_mask])
width_ori = np.mean(w[ori_mask])

cell_x_ori = [-length_ori/2, length_ori/2, length_ori/2, -length_ori/2, -length_ori/2]
cell_y_ori = [-width_ori/2, -width_ori/2, width_ori/2, width_ori/2, -width_ori/2]

length_ter = np.mean(l[ter_mask])
width_ter = np.mean(w[ter_mask])

cell_x_ter = [-length_ter/2, length_ter/2, length_ter/2, -length_ter/2, -length_ter/2]
cell_y_ter = [-width_ter/2, -width_ter/2, width_ter/2, width_ter/2, -width_ter/2]

## Normalizzazione traiettorie

In [73]:
x_real_z = zscore(x_array, axis=1)
y_real_z = zscore(y_array, axis=1)

x_fbm_z = zscore(x_fbm, axis=1)
y_fbm_z = zscore(y_fbm, axis=1)

x_ctrw_z = zscore(x_ctrw, axis=1)
y_ctrw_z = zscore(y_ctrw, axis=1)

Analisi del comportamento diffusivo: n un processo diffusivo normale (moto browniano), il MSD cresce linearmente con tau (MSD(𝜏)∝𝜏). Un esponente α vicino a 1 indica una diffusione normale. Se 𝛼<1, suggerisce una diffusione subdiffusiva; se 𝛼>1, una diffusione superdiffusiva.

## Tipo di moto diffusivo

I dati sembrano essere compatibili con entrambi i modelli - DA STUDIARE MEGLIO

In [ ]:
msd_time_avg = np.mean(msd_loci, axis=1)
msd_ensemble_avg = np.mean(msd_loci, axis=0)

plt.plot(taus, msd_ensemble_avg, label='MSD Ensemble', color='blue')
plt.plot(taus, msd_time_avg[:len(taus)], label='MSD Time-averaged (esempio)', color='orange')
plt.legend()
plt.xscale('log')
plt.yscale('log')
plt.show()

## Esponenti di diffusione delle singole traiettorie ||FUNZIONA||

### Identificazione traiettorie anomale

In [ ]:
threshold_high = msd_loci_mean + 5 * msd_std
threshold_low = msd_loci_mean - 5 * msd_std

anomali_indices = np.where((msd_loci > threshold_high).any(axis=1) | 
                           (msd_loci < threshold_low).any(axis=1))[0]

print(f"Trovate {len(anomali_indices)} traiettorie anomale su {len(msd_loci)} totali, ovvero il {(len(anomali_indices)/len(msd_loci))*100}%")

Traiettorie anomale ori

In [ ]:
threshold_high_ori = msd_loci_mean_ori + 5 * msd_std_ori
threshold_low_ori = msd_loci_mean_ori - 5 * msd_std_ori

anomali_indices_ori = np.where((msd_loci_ori > threshold_high_ori).any(axis=1) | 
                           (msd_loci_ori < threshold_low_ori).any(axis=1))[0]

print(f"Trovate {len(anomali_indices_ori)} traiettorie anomale su {len(msd_loci_ori)} totali, ovvero il {(len(anomali_indices_ori)/len(msd_loci_ori))*100}%")

Traiettorie anomale ter

In [ ]:
threshold_high_ter = msd_loci_mean_ter + 5 * msd_std_ter
threshold_low_ter = msd_loci_mean_ter - 5 * msd_std_ter

anomali_indices_ter = np.where((msd_loci_ter > threshold_high_ter).any(axis=1) | 
                           (msd_loci_ter < threshold_low_ter).any(axis=1))[0]

print(f"Trovate {len(anomali_indices_ter)} traiettorie anomale su {len(msd_loci_ter)} totali, ovvero il {(len(anomali_indices_ter)/len(msd_loci_ter))*100}%")

## Distibuzione degli step

In [ ]:
dx = x_array[:, 1:] - x_array[:, :-1]
dy = y_array[:, 1:] - y_array[:, :-1]
step_lengths = np.sqrt(dx**2 + dy**2)

step_lengths_flat = step_lengths.flatten()

In [ ]:
dx_ori = x_ori[:, 1:] - x_ori[:, :-1]
dy_ori = y_ori[:, 1:] - y_ori[:, :-1]
step_lengths_ori = np.sqrt(dx_ori**2 + dy_ori**2)

step_lengths_flat_ori = step_lengths_ori.flatten()

In [ ]:
dx_ter = x_ter[:, 1:] - x_ter[:, :-1]
dy_ter = y_ter[:, 1:] - y_ter[:, :-1]
step_lengths_ter = np.sqrt(dx_ter**2 + dy_ter**2)

step_lengths_flat_ter = step_lengths_ter.flatten()

## Modello subdiffusivo - Fractional Brownian Motion ||FUNZIONA||

In [ ]:
taus, msd_fbm = calculate_msd(t_array, x_fbm, y_fbm)
msd_mean_fbm = np.mean(msd_fbm, axis=0)
msd_std_fbm = np.std(msd_fbm)
msd_sem_fbm = msd_std_fbm / np.sqrt(msd_fbm.shape[0])

log_taus = np.log(taus)
log_msd_fbm = np.log(msd_mean_fbm)
md_slope_fbm, md_intercept_fbm, md_r_value_fbm, md_p_value_fbm, md_std_err_fbm = stats.linregress(log_taus, log_msd_fbm)

## Modello subdiffusivo - Continuous-Time Random Walk ||FUNZIONA||

In [ ]:
taus, msd_ctrw = calculate_msd(t_array, x_ctrw, y_ctrw)
msd_mean_ctrw = np.mean(msd_ctrw, axis=0)
msd_std_ctrw = np.std(msd_ctrw)
msd_sem_ctrw = msd_std_ctrw / np.sqrt(msd_mean_ctrw.shape[0])

log_taus = np.log(taus)
log_msd_ctrw = np.log(msd_mean_ctrw)
md_slope_ctrw, md_intercept_ctrw, md_r_value_ctrw, md_p_value_ctrw, md_std_err_ctrw = stats.linregress(log_taus, log_msd_ctrw)

# Grafici

## Test di compatibilità dei dati con i modelli - p value - NON FUNZIONA

In [ ]:
p_values_fbm = []
p_values_ctrw = []

for i in range(len(taus)):
    t_stat_fbm, p_fbm = ttest_ind(msd_real_norm[:, i], msd_fbm_norm[:, i], equal_var=False)
    t_stat_ctrw, p_ctrw = ttest_ind(msd_real_norm[:, i], msd_ctrw_norm[:, i], equal_var=False)

    p_values_fbm.append(p_fbm)
    p_values_ctrw.append(p_ctrw)


plt.figure(figsize=(8, 4))
plt.plot(taus, p_values_fbm, marker='o', color='purple', label='Simulazione fbm')
plt.plot(taus, p_values_ctrw, marker='o', color='green', label='Simulazione ctrw')
plt.axhline(0.05, linestyle='--', color='red', label='Significatività 0.05')
plt.xlabel("Tau")
plt.ylabel("p-value")
plt.title("Test t: Differenza tra MSD reali e simulati")
plt.legend()
plt.grid(True)
plt.show()

p_values_ctrw

## Moto (sub)diffusivo - UNICA COSA DECENTE DI QUESTA ANALISI

Giusto in linea generale, ma decisamente da migliorare:
- barre d'errore: non so cosa fare

In [ ]:
plt.figure(figsize=(8, 6))

plt.plot(log_taus, md_slope * log_taus + md_intercept, color='crimson', alpha=0.5, label="Regr. Lineare log(MSD medio)")
plt.plot(log_taus, md_slope_ori * log_taus + md_intercept_ori, color='gold', alpha=0.5, label="Regr. Lineare log(MSD medio ori)")
plt.plot(log_taus, md_slope_ter * log_taus + md_intercept_ter, color='steelblue', alpha=0.5, label="Regr. Lineare log(MSD medio ter)")

plt.plot(log_taus, log_msd, label="log(MSD medio)", color="crimson", linestyle='-', marker='o')
plt.plot(log_taus, log_msd_ori, label="log(MSD medio ori)", color="gold", linestyle='-', marker='o')
plt.plot(log_taus, log_msd_ter, label="log(MSD medio ter)", color="steelblue", linestyle='-', marker='o')

plt.xlabel('log(tempo)', fontsize=10)
plt.ylabel('log(MSD)', fontsize=10)
plt.title('Moto subdiffusivo', fontsize=10)

plt.legend(fontsize=10)

plt.grid(True)
plt.show()

## Distribuzione degli esponenti di diffusione - TUTTO MOLTO BELLO, MA MI SERVE?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharex=True, sharey=True)

axes[0].hist(alpha_values, bins=40, alpha=0.7, color='crimson', edgecolor='black', density=True)
axes[0].axvline(np.mean(alpha_values), color='black', linestyle='dashed', label=f'Media: {np.mean(alpha_values):.2f}')
axes[0].set_title("Distribuzione α", fontsize=14)
axes[0].set_xlabel("Esponente di diffusione α", fontsize=12)
axes[0].set_ylabel("Densità di probabilità", fontsize=12)
axes[0].legend()

# Istogramma per "Ori"
axes[1].hist(alpha_ori, bins=40, alpha=0.7, color='gold', edgecolor='black', density=True)
axes[1].axvline(np.mean(alpha_ori), color='black', linestyle='dashed', label=f'Media Ori: {np.mean(alpha_ori):.2f}')
axes[1].set_title("Distribuzione α - Ori", fontsize=14)
axes[1].set_xlabel("Esponente di diffusione α", fontsize=12)
axes[1].legend()

# Istogramma per "Ter"
axes[2].hist(alpha_ter, bins=40, alpha=0.7, color='steelblue', edgecolor='black', density=True)
axes[2].axvline(np.mean(alpha_ter), color='black', linestyle='dashed', label=f'Media Ter: {np.mean(alpha_ter):.2f}')
axes[2].set_title("Distribuzione α - Ter", fontsize=14)
axes[2].set_xlabel("Esponente di diffusione α", fontsize=12)
axes[2].legend()

plt.tight_layout()
plt.show()


## Traiettorie anomale con MSD - A ME PIACE, AL PROF NO

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(12, 15))

for idx in anomali_indices:
    axes[0][0].plot(x_array[idx], y_array[idx], alpha=0.7)
    
axes[0][0].plot(cell_x, cell_y, 'k-', label="Parete cellulare media")

axes[0][0].set_xlabel("Posizione X", fontsize=10)
axes[0][0].set_ylabel("Posizione Y", fontsize=10)
axes[0][0].set_title("Traiettorie Anomale", fontsize=10)
axes[0][0].legend()
axes[0][0].grid(True)

for idx in anomali_indices:
    axes[0][1].plot(taus, msd_loci[idx], alpha=0.7)

axes[0][1].plot(taus, msd_loci_mean, color="black", linestyle="dashed", linewidth=2)
axes[0][1].fill_between(taus, threshold_low, threshold_high, color='pink', alpha=0.3, label="Intervallo ±5σ")

axes[0][1].set_xlabel("Tau (ritardo temporale)", fontsize=10)
axes[0][1].set_ylabel("MSD", fontsize=10)
axes[0][1].set_title("MSD delle Traiettorie Anomale", fontsize=10)
axes[0][1].legend()
axes[0][1].grid(True)

for idx in anomali_indices_ori:
    axes[1][0].plot(x_ori[idx], y_ori[idx], alpha=0.7)

axes[1][0].plot(cell_x_ori, cell_y_ori, 'k-', label="Parete cellulare media")

axes[1][0].set_xlabel("Posizione X", fontsize=10)
axes[1][0].set_ylabel("Posizione Y", fontsize=10)
axes[1][0].set_title("Traiettorie Anomale Ori", fontsize=10)
axes[1][0].legend()
axes[1][0].grid(True)

for idx in anomali_indices_ori:
    axes[1][1].plot(taus, msd_loci_ori[idx], alpha=0.7)

axes[1][1].plot(taus, msd_loci_mean_ori, color="black", linestyle="dashed", linewidth=2)
axes[1][1].fill_between(taus, threshold_low_ori, threshold_high_ori, color='pink', alpha=0.3, label="Intervallo ±5σ")

axes[1][1].set_xlabel("Tau (ritardo temporale)", fontsize=10)
axes[1][1].set_ylabel("MSD", fontsize=10)
axes[1][1].set_title("MSD delle Traiettorie Anomale Ori", fontsize=10)
axes[1][1].legend()
axes[1][1].grid(True)

for idx in anomali_indices_ter:
    axes[2][0].plot(x_ter[idx], y_ter[idx], alpha=0.7)

axes[2][0].plot(cell_x_ter, cell_y_ter, 'k-', label="Parete cellulare media")

axes[2][0].set_xlabel("Posizione X", fontsize=10)
axes[2][0].set_ylabel("Posizione Y", fontsize=10)
axes[2][0].set_title("Traiettorie Anomale Ter", fontsize=10)
axes[2][0].legend()
axes[2][0].grid(True)

for idx in anomali_indices_ter:
    axes[2][1].plot(taus, msd_loci_ter[idx], alpha=0.7)

axes[2][1].plot(taus, msd_loci_mean_ter, color="black", linestyle="dashed", linewidth=2)
axes[2][1].fill_between(taus, threshold_low_ter, threshold_high_ter, color='pink', alpha=0.3, label="Intervallo ±5σ")

axes[2][1].set_xlabel("Tau (ritardo temporale)", fontsize=10)
axes[2][1].set_ylabel("MSD", fontsize=10)
axes[2][1].set_title("MSD delle Traiettorie Anomale Ter", fontsize=10)
axes[2][1].legend()
axes[2][1].grid(True)

plt.tight_layout()
plt.show()

## Distribuzione degli step - BOH VIENE BRUTTINA E NON SO SE HA SENSO

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 5))

axes[0].hist(step_lengths_flat, bins=100, color='crimson', alpha=0.8, density=True)
axes[1].hist(step_lengths_flat_ori, bins=100, color='gold', alpha=0.8, density=True)
axes[2].hist(step_lengths_flat_ter, bins=100, color='steelblue', alpha=0.8, density=True)
plt.xlabel('Lunghezza dello step')
plt.ylabel('Densità')
plt.title('Distribuzione delle lunghezze degli step')
plt.show()